# Notebook 07: Ship Mode Performance Analysis

## Objective
This notebook evaluates shipping performance across different ship modes.

### Key Tasks:
- Compare average lead time by ship mode
- Analyze shipment volume by ship mode
- Calculate delay frequency
- Identify best and worst performing ship modes
- Evaluate descriptive cost-time tradeoffs

This step helps understand how shipping choices impact efficiency and delays.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [3]:
# Load dataset
ncd = pd.read_csv("../data/featured_nassau_data.csv")

# Ensure datetime
ncd["Order Date"] = pd.to_datetime(ncd["Order Date"])
ncd["Ship Date"] = pd.to_datetime(ncd["Ship Date"])

ncd.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost,Lead_Time_Days,Shipping Lead Time,Factory,Route_State,Route_Region,Order Month,Order Year,Order Month Name,Delay Status,Shipping Speed Category
0,1,US-2021-103800-CHO-MIL-31000,2024-01-03,2024-01-09,Standard Class,103800,United States,Houston,Texas,77095,Chocolate,Interior,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28,6,6,Wicked Choccy's,Wicked Choccy's → Texas,Wicked Choccy's → Interior,1,2024,January,Delayed,Slow
1,2,US-2021-112326-CHO-TRI-54000,2024-01-04,2024-01-11,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60,7,7,Wicked Choccy's,Wicked Choccy's → Illinois,Wicked Choccy's → Interior,1,2024,January,Delayed,Slow
2,3,US-2021-112326-CHO-NUT-13000,2024-01-04,2024-01-09,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00,5,5,Lot's O' Nuts,Lot's O' Nuts → Illinois,Lot's O' Nuts → Interior,1,2024,January,On-Time,Slow
3,4,US-2021-112326-CHO-SCR-58000,2024-01-04,2024-01-11,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30,7,7,Lot's O' Nuts,Lot's O' Nuts → Illinois,Lot's O' Nuts → Interior,1,2024,January,Delayed,Slow
4,5,US-2021-141817-CHO-TRI-54000,2024-01-05,2024-01-12,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90,7,7,Wicked Choccy's,Wicked Choccy's → Pennsylvania,Wicked Choccy's → Atlantic,1,2024,January,Delayed,Slow


## Ship Mode Performance Summary

In [4]:
ship_mode_summary = ncd.groupby("Ship Mode").agg(
    Total_Shipments=("Order ID", "count"),
    Avg_Lead_Time=("Shipping Lead Time", "mean"),
    Lead_Time_Std=("Shipping Lead Time", "std")
).reset_index()

ship_mode_summary["Avg_Lead_Time"] = ship_mode_summary["Avg_Lead_Time"].round(2)
ship_mode_summary["Lead_Time_Std"] = ship_mode_summary["Lead_Time_Std"].round(2)

ship_mode_summary

,Ship Mode,Total_Shipments,Avg_Lead_Time,Lead_Time_Std
0,First Class,1548,2.02,0.82
1,Same Day,547,0.50,0.50
2,Second Class,1979,3.48,1.13
3,Standard Class,6120,5.00,1.42


## Delay Frequency by Ship Mode

In [5]:
delay_analysis = ncd.groupby(["Ship Mode", "Delay Status"]).size().unstack()

delay_analysis

Delay Status,Delayed,On-Time
Ship Mode,,
First Class,NaN,1548.0
Same Day,NaN,547.0
Second Class,NaN,1979.0
Standard Class,2456.0,3664.0


In [6]:
# Convert to percentage
delay_percent = delay_analysis.div(delay_analysis.sum(axis=1), axis=0) * 100
delay_percent.round(2)

Delay Status,Delayed,On-Time
Ship Mode,,
First Class,NaN,100.00
Same Day,NaN,100.00
Second Class,NaN,100.00
Standard Class,40.13,59.87


## Combine Metrics

In [7]:
# Merge summary + delay
delay_percent.reset_index(inplace=True)

ship_mode_final = ship_mode_summary.merge(delay_percent, on="Ship Mode")

ship_mode_final

,Ship Mode,Total_Shipments,Avg_Lead_Time,Lead_Time_Std,Delayed,On-Time
0,First Class,1548,2.02,0.82,NaN,100.000000
1,Same Day,547,0.50,0.50,NaN,100.000000
2,Second Class,1979,3.48,1.13,NaN,100.000000
3,Standard Class,6120,5.00,1.42,40.130719,59.869281


## Rank Ship Modes

In [8]:
# Rank by efficiency (lower lead time = better)
ship_mode_final["Efficiency_Rank"] = ship_mode_final["Avg_Lead_Time"].rank()

ship_mode_final.sort_values("Efficiency_Rank")

,Ship Mode,Total_Shipments,Avg_Lead_Time,Lead_Time_Std,Delayed,On-Time,Efficiency_Rank
1,Same Day,547,0.50,0.50,NaN,100.000000,1.0
0,First Class,1548,2.02,0.82,NaN,100.000000,2.0
2,Second Class,1979,3.48,1.13,NaN,100.000000,3.0
3,Standard Class,6120,5.00,1.42,40.130719,59.869281,4.0


## Business Interpretation

- Ship modes with lower average lead time are more efficient
- Ship modes with higher delay percentage indicate reliability issues
- High volume + high delay = operational risk
- Trade-off exists between speed and cost

This analysis helps determine which shipping strategy should be prioritized.

## Final Check

In [9]:
ship_mode_final

,Ship Mode,Total_Shipments,Avg_Lead_Time,Lead_Time_Std,Delayed,On-Time,Efficiency_Rank
0,First Class,1548,2.02,0.82,NaN,100.000000,2.0
1,Same Day,547,0.50,0.50,NaN,100.000000,1.0
2,Second Class,1979,3.48,1.13,NaN,100.000000,3.0
3,Standard Class,6120,5.00,1.42,40.130719,59.869281,4.0


## Save Output

In [10]:
ship_mode_final.to_csv("../data/ship_mode_detailed_analysis.csv", index=False)

print("Ship mode analysis saved successfully.")

Ship mode analysis saved successfully.
